## Setup

Keep [`example_data.py`](example_data.py) and [`privacy_metrics.py`](privacy_metrics.py) beside this notebook.

In [1]:
#| label: setup
from IPython.display import display

from example_data import make_example_views
from privacy_metrics import exact_match_indicators, exact_match_rate

## 1. Metric and inputs

| Property | Value |
|---|---|
| Metric | Exact-match rate |
| Family | Privacy |
| Origin | Established and adapted in the source project |
| Classifier | None |
| Input | $\widetilde X,\widetilde Y$ |
| Holdout | None |
| Range | $[0,1]$; smaller is better for privacy |

Here $X$ is the reference or training table and $Y$ is the released comparison table. Their roles are not interchangeable.

## 2. Mathematics

For row $\widetilde y_i$ of $\widetilde Y$, define

$$
e_i=\mathbf1\left\{\exists k:\widetilde y_i=\widetilde x_k\right\},
$$

where equality means equality in every transformed column. The exact-match rate is

$$
D_{\mathrm{exact}}(X,Y)=\frac1{n_Y}\sum_{i=1}^{n_Y}e_i.
$$

Each row of $Y$ contributes separately, so repeated copies are counted repeatedly. The source implementation rounds transformed values to nine decimal places before hashing rows. Missing values match only when their indicators and explicit missing category also match.

In [2]:
#| label: exact-match

example = make_example_views()
X_tilde, Y_tilde = example.X_tilde, example.Y_tilde

indicators = exact_match_indicators(X_tilde, Y_tilde)
D_exact = exact_match_rate(X_tilde, Y_tilde)
result = Y_tilde.copy()
result["matches a row of X"] = indicators

display(result)
print(f"Exact-match rate = {D_exact:.6f}")

,age,age__missing,income_k,income_k__missing,region__code,outcome,matches a row of X
y1,0.027778,0,0.050847,0,1,0,False
y2,0.416667,1,0.322034,0,2,1,False
y3,0.555556,0,0.389831,1,3,1,False
y4,1.000000,0,0.949153,0,4,1,False
y5,0.361111,0,0.542373,0,0,0,False


Exact-match rate = 0.000000


### Sanity check with one deliberate copy

The shared $Y$ contains no exact copy. To verify the definition, replace its first row by the first row of $X$ without changing the metric code.

In [3]:
#| label: deliberate-copy

Y_with_copy = Y_tilde.copy()
Y_with_copy.iloc[0] = X_tilde.iloc[0]
copy_rate = exact_match_rate(X_tilde, Y_with_copy)
print(f"Rate after inserting one copy among {len(Y_with_copy)} rows = {copy_rate:.6f}")
assert copy_rate == 1 / len(Y_with_copy)

Rate after inserting one copy among 5 rows = 0.200000


## 3. Interpretation

Zero means no row of $Y$ exactly matches a row of $X$; one means every row of $Y$ matches at least one row of $X$. The measure is asymmetric because its denominator is $n_Y$.

A zero rate does not prove privacy: nearly copied rows are ignored. The distance-based membership metric in the next lesson addresses closeness rather than exact equality.

## Takeaway

For every released row in $Y$, ask whether the complete row appears anywhere in $X$, then average those binary answers over $Y$.